# `sample_bounded_fluxes` — Step-by-step walkthrough

This notebook dissects every step of `bounded_fluxes.sample_bounded_fluxes`, verifying each sub-computation individually and using `scipy.optimize.root` instead of Newton's method for final refinement — making the solution procedure transparent and easy to cross-check.

**Outline**
1. [Model and sampler setup](#1)
2. [Step 1 — Global eigenvalue bounds](#2)
3. [Step 2 — Bounding box for NSNS-flux $h$](#3)
4. [Step 3 — ISD completion: $h \mapsto f$](#4)
5. [Step 4 — Tadpole and bound filters](#5)
6. [Step 5 — ISD convention consistency check](#6)
7. [Step 6 — Refinement with `scipy.optimize.root`](#7)
8. [Summary and comparison](#8)

*(Created: Andreas Schachner, 2025)*

## Imports

In [ ]:
# General imports
import warnings
import numpy as np
from scipy.optimize import root
from tqdm.auto import tqdm

# JAX imports
import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)

# Plotting
import matplotlib.pyplot as plt

# JAXVacua
import jaxvacua as jvc
from jaxvacua.flux_bounding import bounded_fluxes

warnings.filterwarnings('ignore')

<a id="1"></a>
## 1. Model and sampler setup

We use the same $h^{1,2}=2$ Kreuzer–Skarke geometry as in notebooks 10 and 10b.
The moduli region is:

$$
    \mathrm{Im}(z_i) \in [2, 5],\quad
    s = \mathrm{Im}(\tau) \in [2, 10],\quad
    c_0 = \mathrm{Re}(\tau) \in [-0.5, 0.5].
$$

We set $N_{\max} = 5$ so that the bounding box is small enough to enumerate exhaustively as a cross-check.

In [ ]:
h12 = 2
model = jvc.FluxVacuaFinder(h12=h12,model_ID=1)

model

In [ ]:
DILATON_BOUNDS  = [2, 10]
MODULI_BOUNDS   = [2, 5]
AXION_BOUNDS    = [-0.5, 0.5]
NMAX            = 2

sampler = jvc.data_sampler(
    model,
    flux_bounds=[-5, 5],
    moduli_bounds=MODULI_BOUNDS,
    axion_bounds=AXION_BOUNDS,
    dilaton_bounds=DILATON_BOUNDS,
    seed=0,
)

print(f"s   ∈ [{sampler.s_lower}, {sampler.s_upper}]")
print(f"Im(z) ∈ [{sampler.moduli_lower}, {sampler.moduli_upper}]")
print(f"c₀  ∈ [{sampler.axion_lower}, {sampler.axion_upper}]")

<a id="2"></a>
## 2. Step 1 — Global eigenvalue bounds

`sample_bounded_fluxes` first samples $n_{\mathrm{sample}}$ moduli points, computes three eigenvalue quantities at each point, and takes global extrema.

| Quantity | Matrix | Role |
|----------|--------|------|
| $\lambda_{\max}$ | ISD matrix $\mathcal{M}$ | bounds $\|h\|^2$ and $\|f\|^2$ |
| $\mu_{\min/\max}$ | $-\mathrm{Im}(\mathcal{N})$ | bounds $\|h_2\|^2$ (B-cycle NSNS sector) |
| $\tilde{\mu}_{\min/\max}$ | $\mathrm{Im}(\mathcal{N}^{-1})$ | bounds $\|h_1\|^2$ (A-cycle NSNS sector) |

Below we reproduce this step manually so we can inspect every number.

In [ ]:
N_SAMPLE = 300
np.random.seed(42)

moduli_sample = sampler.get_complex_moduli(N_SAMPLE)
tau_sample    = sampler.get_complex_tau(N_SAMPLE)

print(f"Sampled {len(moduli_sample)} moduli points.")
print(f"Im(z) range: [{moduli_sample.imag.min():.3f}, {moduli_sample.imag.max():.3f}]")
print(f"s range:     [{tau_sample.imag.min():.3f}, {tau_sample.imag.max():.3f}]")

In [ ]:
# Compute eigenvalues at a single representative moduli point.
z0  = moduli_sample[0]
tau0 = tau_sample[0]
z0c = jnp.conj(z0)

# Gauge kinetic matrix N(z, z_bar)
N_mat = model.gauge_kinetic_matrix(z0, z0c)
print("N matrix (gauge kinetic):")
print(N_mat)

# Eigenvalues of -Im(N)
mu_evs = jnp.linalg.eigvalsh(-N_mat.imag)
print(f"\nEigenvalues of -Im(N): {np.array(mu_evs)}")
print(f"  μ_min = {float(mu_evs.min()):.6f},  μ_max = {float(mu_evs.max()):.6f}")

# Eigenvalues of Im(N^{-1})
tmu_evs = jnp.linalg.eigvalsh(jnp.linalg.inv(N_mat).imag)
print(f"\nEigenvalues of Im(N⁻¹): {np.array(tmu_evs)}")
print(f"  μ̃_min = {float(tmu_evs.min()):.6f},  μ̃_max = {float(tmu_evs.max()):.6f}")

In [ ]:
# ISD matrix M(z, z_bar) — formula from periods.py:
#   Block1 = [Im(N) + Re(N) @ Im(N)^{-1} @ Re(N)  ,  Re(N) @ Im(N)^{-1}]
#   Block2 = [Im(N)^{-1} @ Re(N)                   ,  Im(N)^{-1}         ]
#   M = -[Block1; Block2]

M0 = model.ISD_matrix(z0, z0c)
lm_evs = jnp.linalg.eigvalsh(M0)
print("ISD matrix M shape:", M0.shape)
print(f"ISD matrix eigenvalues: {np.array(lm_evs)}")
print(f"  λ_max = {float(lm_evs.max()):.6f}")
print()
# Verify M is positive definite
print(f"All eigenvalues positive? {bool(jnp.all(lm_evs > 0))}")

In [ ]:
# Now compute global extrema over the full moduli sample using the JIT-compiled vmap.
# This is what compute_bounding_box does internally.
bf = bounded_fluxes(model, sampler=sampler, Nmax=NMAX)

# The dil_min should tighten from sqrt(3)/2 to sampler.s_lower=2
# (done inside sample_bounded_fluxes / enumerate_fluxes automatically)
bf.dil_min = float(sampler.s_lower)  # manual tightening for this walkthrough
print(f"dil_min (used for bounds) = {bf.dil_min}  (= sampler.s_lower)")

# JIT-vmapped eigenvalue computation over all moduli points
evs_all, M_all = bf._compute_evs_and_M_vmap(jnp.array(moduli_sample, dtype=complex))
lm_arr, mu_min_arr, mu_max_arr, tmu_min_arr, tmu_max_arr = evs_all

lambda_max_gl   = float(jnp.max(lm_arr))
mu_min_gl       = float(jnp.min(mu_min_arr))
mu_max_gl       = float(jnp.max(mu_max_arr))
tmu_min_gl      = float(jnp.min(tmu_min_arr))
tmu_max_gl      = float(jnp.max(tmu_max_arr))

print(f"\nGlobal eigenvalue extrema over {N_SAMPLE} moduli points:")
print(f"  λ_max   = {lambda_max_gl:.4f}")
print(f"  μ_min   = {mu_min_gl:.4f}")
print(f"  μ_max   = {mu_max_gl:.4f}")
print(f"  μ̃_min  = {tmu_min_gl:.6f}")
print(f"  μ̃_max  = {tmu_max_gl:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, dpi=130, figsize=(13, 3.2))

axes[0].hist(np.array(lm_arr), bins=40, edgecolor='k', lw=0.4)
axes[0].axvline(lambda_max_gl, color='r', lw=1.5, label=f'λ_max={lambda_max_gl:.1f}')
axes[0].set_xlabel(r'$\lambda_{\max}(\mathcal{M})$', fontsize=11)
axes[0].set_ylabel('Count', fontsize=11)
axes[0].set_title('ISD matrix max eigenvalue', fontsize=10)
axes[0].legend(fontsize=8)

axes[1].hist(np.array(mu_min_arr), bins=40, edgecolor='k', lw=0.4)
axes[1].axvline(mu_min_gl, color='r', lw=1.5, label=f'μ_min={mu_min_gl:.3f}')
axes[1].set_xlabel(r'$\mu_{\min}(-\mathrm{Im}\,\mathcal{N})$', fontsize=11)
axes[1].set_title(r'Gauge kinetic $\mu_{\min}$', fontsize=10)
axes[1].legend(fontsize=8)

axes[2].hist(np.array(tmu_min_arr), bins=40, edgecolor='k', lw=0.4)
axes[2].axvline(tmu_min_gl, color='r', lw=1.5, label=f'μ̃_min={tmu_min_gl:.5f}')
axes[2].set_xlabel(r'$\tilde\mu_{\min}(\mathrm{Im}\,\mathcal{N}^{-1})$', fontsize=11)
axes[2].set_title(r'Inverse gauge kinetic $\tilde\mu_{\min}$', fontsize=10)
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

<a id="3"></a>
## 3. Step 2 — Bounding box for NSNS-flux $h$

From the inequalities in arXiv:2501.03984 the allowed integer NSNS-fluxes $h = (h_1, h_2)$ must satisfy:

$$
    \tilde\mu_{\min}\,\|h_1\|^2 \leq \frac{N_{\max}}{s_{\min}},\qquad
    \mu_{\min}\,\|h_2\|^2 \leq \frac{N_{\max}}{s_{\min}},\qquad
    \|h\|^2 \leq \frac{2\,\lambda_{\max}\,N_{\max}}{s_{\min}}.
$$

Here $s_{\min}$ is `dil_min`, which we set to `sampler.s_lower = 2`.

The **key insight** is that $s_{\min}$ must reflect the actual lower bound of the sampled region, not the SL(2,Z) fundamental domain floor $\sqrt{3}/2 \approx 0.866$. Using the correct $s_{\min} = 2$ tightens the box radii by a factor $\sqrt{2/0.866} \approx 1.5$, reducing box volume by $\sim 3.4\times$ for each component.

In [ ]:
s_min = bf.dil_min  # = 2.0 after tightening

h1_box = np.sqrt(NMAX / (s_min * tmu_min_gl))
h2_box = np.sqrt(NMAX / (s_min * mu_min_gl))
h_box  = np.sqrt(2.0 * lambda_max_gl * NMAX / s_min)

print(f"Bounding box (with s_min = {s_min}):")
print(f"  h1_box = sqrt(Nmax / (s_min * μ̃_min)) = sqrt({NMAX} / ({s_min} * {tmu_min_gl:.5f})) = {h1_box:.3f}")
print(f"  h2_box = sqrt(Nmax / (s_min * μ_min))  = sqrt({NMAX} / ({s_min} * {mu_min_gl:.5f})) = {h2_box:.3f}")
print(f"  h_box  = sqrt(2*λ_max*Nmax / s_min)    = sqrt(2*{lambda_max_gl:.3f}*{NMAX} / {s_min})  = {h_box:.3f}")

# Cross-check against compute_bounding_box
h1_box_cc, h2_box_cc, h_box_cc = bf.compute_bounding_box(
    jnp.array(moduli_sample, dtype=complex)
)
print(f"\nFrom compute_bounding_box:")
print(f"  h1_box={h1_box_cc:.3f}, h2_box={h2_box_cc:.3f}, h_box={h_box_cc:.3f}")
print(f"  (matches: {np.allclose([h1_box, h2_box, h_box], [h1_box_cc, h2_box_cc, h_box_cc], rtol=1e-4)})")

In [ ]:
# What would happen if we forgot to tighten dil_min?
s_min_wrong = np.sqrt(3) / 2  # SL(2,Z) fundamental domain floor
h1_box_wrong = np.sqrt(NMAX / (s_min_wrong * tmu_min_gl))
h2_box_wrong = np.sqrt(NMAX / (s_min_wrong * mu_min_gl))
h_box_wrong  = np.sqrt(2.0 * lambda_max_gl * NMAX / s_min_wrong)

print(f"Comparison of bounding box radii:")
print(f"  s_min = {s_min_wrong:.4f} (wrong, SL(2,Z) floor):  h_box = {h_box_wrong:.2f}")
print(f"  s_min = {s_min:.4f} (correct, sampler floor): h_box = {h_box:.2f}")
print(f"  Volume ratio: ({h_box_wrong:.2f}/{h_box:.2f})^4 ≈ {(h_box_wrong/h_box)**4:.1f}x more candidates with wrong s_min")

In [ ]:
# Enumerate all integer h vectors inside the bounding box (for small Nmax=2 this is feasible)

h_candidates = bf.get_h_candidates()
print(f"Total integer h candidates inside bounding box: {len(h_candidates)}")
print(f"h vector shape: {h_candidates.shape}  (n_candidates x n_fluxes)")
print(f"n_fluxes = {model.n_fluxes}, dimension_H3 = {model.dimension_H3}")
print(f"  → h = [h₁ | h₂] where h₁,h₂ each have {model.dimension_H3} components")


<a id="4"></a>
## 4. Step 3 — ISD completion: $h \mapsto f$

For each NSNS-flux $h$ and each sampled moduli point $(z, \tau)$, we compute the ISD-projected RR-flux:

$$
    f = s\,\mathcal{M}(z,\bar z)\,\Sigma\,h + c_0\,h,
    \qquad \tau = c_0 + \mathrm{i}s.
$$

This is mode `"H"` in `sampling.py`'s `_ISD_sampling_FH`. The result $f$ is **continuous** (not integer); we round to the nearest integer and later check the tadpole constraint.

### Convention verification

We verify that `flux_bounding.py`'s formula matches `sampling.py`'s formula exactly.

In [ ]:
# Take the first moduli point as our working example
z_ex  = jnp.array(moduli_sample[0], dtype=complex)
tau_ex = jnp.array(tau_sample[0], dtype=complex)
z_ex_c = jnp.conj(z_ex)
tau_ex_c = jnp.conj(tau_ex)

c0_ex = float(jnp.real(tau_ex))
s_ex  = float(jnp.imag(tau_ex))

print(f"Working moduli point:")
print(f"  z   = {np.array(z_ex)}")
print(f"  τ   = {complex(tau_ex):.6f}  (c₀={c0_ex:.4f}, s={s_ex:.4f})")

# Symplectic form sigma from model
sigma = model.periods.sigma
print(f"\nSymplectic form σ:")
print(np.array(sigma).astype(int))

In [ ]:
# Pick a sample h vector
h_test = jnp.array([0, 1, 2, 0, 0, 1], dtype=float)  # [h1_1, h1_2, h2_1, h2_2]

# Method A: flux_bounding.py formula
M0_ex = model.ISD_matrix(z_ex, z_ex_c)
M0_sigma = M0_ex @ sigma
f_bounding = s_ex * (M0_sigma @ h_test) + c0_ex * h_test

# Method B: sampling.py _ISD_sampling_FH mode="H" formula
SigmaFlux = sigma @ h_test              # σ @ h
M0_SigmaFlux = M0_ex @ SigmaFlux        # M₀ @ σ @ h
f_sampling = M0_SigmaFlux * s_ex + h_test * c0_ex

# Method C: sampling.py's ISD_sampling function (the official API)
#flux_full_test = jnp.concatenate([jnp.zeros(model.n_fluxes), h_test])  # [f=0 | h]
flux_full_test = h_test
f_official = sampler.ISD_sampling(
    z_ex, z_ex_c, tau_ex, tau_ex_c,
    flux_full_test,
    mode="H", output="half", return_integer_flux=False,
)

print("ISD completion: f = s * M₀ @ (σ @ h) + c₀ * h")
print(f"  f (flux_bounding formula) = {np.array(f_bounding)}")
print(f"  f (sampling.py formula)   = {np.array(f_sampling)}")
print(f"  f (ISD_sampling API)      = {np.array(f_official)}")
print()
print(f"  All agree: {np.allclose(f_bounding, f_sampling) and np.allclose(f_bounding, f_official)}")

In [ ]:
# Batch ISD completion: apply to all h candidates in parallel
# This is the inner loop of _process_h_at_modulus_jit

h_chunk = jnp.array(h_candidates[:200], dtype=float)  # first 200 for display

# Batched formula: (M0_sigma @ h_chunk.T).T  applies M0_sigma to each h as a column
f_chunk = s_ex * (M0_sigma @ h_chunk.T).T + c0_ex * h_chunk
f_int_chunk = jnp.round(f_chunk)  # round to nearest integer

print(f"ISD-completed f for first 5 h candidates (before rounding):")
for i in range(5):
    print(f"  h = {np.array(h_candidates[i])}, f_cont = {np.array(f_chunk[i]).round(3)}, f_int = {np.array(f_int_chunk[i]).astype(int)}")

<a id="5"></a>
## 4. Step 4 — Tadpole and eigenvalue bound filters

After ISD completion we apply two filters to each candidate flux $[f | h]$:

1. **Tadpole constraint**: $N_{\mathrm{flux}} = |f^T \Sigma\, h| \in (0, N_{\max}]$
2. **Eigenvalue bounds**: a set of local and global inequalities from arXiv:2501.03984 involving $\|h\|^2, \|f\|^2$, and the eigenvalues at this moduli point.

We verify the tadpole formula first.

In [ ]:
# D3-tadpole formula: N_flux = f @ sigma @ h  (antisymmetric bilinear form)
# For a full flux vector [f | h] of length 2*n_fluxes:
#   N_flux = flux[:n_fl] @ sigma @ flux[n_fl:]

n_fl = model.n_fluxes
dim  = model.dimension_H3

# Use the first ISD-completed example
h_ex = jnp.array(h_candidates[10], dtype=float)
f_ex = jnp.round(s_ex * (M0_sigma @ h_ex) + c0_ex * h_ex)
flux_ex = jnp.concatenate([f_ex, h_ex])

# Compute tadpole three ways:
N1 = float(jnp.abs(f_ex @ sigma @ h_ex).real)
N2 = float(jnp.abs(flux_ex[:n_fl] @ jnp.dot(sigma, flux_ex[n_fl:])).real)
N3 = float(model.tadpole(flux_ex).real)

print(f"Flux example: f = {np.array(f_ex).astype(int)}, h = {np.array(h_ex).astype(int)}")
print(f"Tadpole N_flux = f @ σ @ h:")
print(f"  Method 1 (direct):          {N1:.6f}")
print(f"  Method 2 (via full vector): {N2:.6f}")
print(f"  Method 3 (model.tadpole):   {N3:.6f}")
print(f"  All agree: {np.isclose(N1, N2) and np.isclose(N1, N3)}")

In [ ]:
# For an exactly ISD flux (not rounded), the tadpole has a clean form:
#   N_flux = s * h^T @ M^{-1} @ h  (positive for positive-definite M)

h_ex_f = jnp.array(h_candidates[10], dtype=float)
f_ex_cont = s_ex * (M0_sigma @ h_ex_f) + c0_ex * h_ex_f  # continuous (not rounded)

N_continuous = float(jnp.abs(f_ex_cont @ sigma @ h_ex_f).real)
N_formula    = (s_ex * h_ex_f @ jnp.linalg.inv(M0_ex) @ h_ex_f).real

print("For the CONTINUOUS (pre-rounding) ISD flux:")
print(f"  N_flux = f_cont @ σ @ h        = {N_continuous:.8f}")
print(f"  N_flux = s * h^T M^{{-1}} h     = {N_formula:.8f}")
print(f"  Match (ISD identity):          {np.isclose(N_continuous, N_formula)}")
print()
print("Note: after rounding f to integers, N_flux changes slightly.")

In [ ]:
# Run the full JIT-compiled filter on all h candidates at this one moduli point.
from jaxvacua.flux_bounding import _process_h_at_modulus_jit

evs_ex = (
    float(jnp.max(jnp.linalg.eigvalsh(M0_ex))),
    float(jnp.min(jnp.linalg.eigvalsh(-N_mat.imag))),
    float(jnp.max(jnp.linalg.eigvalsh(-N_mat.imag))),
    float(jnp.min(jnp.linalg.eigvalsh(jnp.linalg.inv(N_mat).imag))),
    float(jnp.max(jnp.linalg.eigvalsh(jnp.linalg.inv(N_mat).imag))),
)

dil_max = float(lambda_max_gl * NMAX)

flux_all, valid_all = _process_h_at_modulus_jit(
    jnp.array(h_candidates, dtype=float),
    M0_sigma,
    n_fl, dim,
    jnp.array(s_ex),
    jnp.array(c0_ex),
    evs_ex,
    tau_ex,
    sigma,
    lambda_max_gl, mu_min_gl, mu_max_gl, tmu_min_gl, tmu_max_gl,
    float(s_min), dil_max, float(NMAX),
)

n_valid = int(jnp.sum(valid_all))
print(f"Candidates passing all filters (at moduli point 0): {n_valid} / {len(h_candidates)}")
print(f"Pass rate: {100*n_valid/len(h_candidates):.2f}%")

In [ ]:
# Manually check the tadpole for all passing candidates
valid_idx = np.where(np.array(valid_all))[0]
flux_valid = np.array(flux_all)[valid_idx]

tadpoles = np.array([float(model.tadpole(jnp.array(fv)).real) for fv in flux_valid])
print(f"Tadpoles of passing candidates:")
print(f"  min={tadpoles.min():.1f}, max={tadpoles.max():.1f}")
print(f"  All in (0, {NMAX}]: {bool(np.all((tadpoles > 0) & (tadpoles <= NMAX)))}")
print()
print(f"First few passing fluxes [f | h] (integer):")
for i, (fv, tad) in enumerate(zip(flux_valid[:5], tadpoles[:5])):
    print(f"  [{i}] {fv.astype(int).tolist()}  N_flux={tad:.0f}")

<a id="6"></a>
## 5. Step 5 — ISD convention consistency check

We perform a direct cross-check: take a single ISD flux constructed by `sampling.py`'s `initial_guesses_ISD`, and verify that:

1. The ISD condition $\star G_3 = \mathrm{i}\,G_3$ is approximately satisfied (up to the integrality rounding)
2. The tadpole $N_{\mathrm{flux}} = f^T \Sigma h$ is positive and within bounds
3. The `flux_bounding.py` formula gives the same $f$ for the same $(h, z, \tau)$

In [ ]:
# ISD condition: f = s * M(z) * sigma * h + c0 * h
# Equivalently: (f - c0*h) = s * M(z) * sigma * h
# Or: ftilde = f - c0*h satisfies ftilde = s * M * sigma * h
#
# Check the degree to which the integer-rounded ISD flux satisfies this.

f_int = jnp.round(f_ex_cont)  # integer-rounded ISD flux
ftilde = f_int - c0_ex * h_ex_f
rhs    = s_ex * (M0_sigma @ h_ex_f)

print("ISD residual check for integer-rounded flux:")
print(f"  f_tilde  = f - c₀ h = {np.array(ftilde).round(4)}")
print(f"  s M σ h             = {np.array(rhs).round(4)}")
print(f"  ||f_tilde - s M σ h|| = {float(jnp.linalg.norm(ftilde - rhs)):.4f}  (rounding error)")

In [ ]:
# Full ISD condition in terms of G3 = f - tau*h (complex flux)
# Self-duality: G3 = (f - tau*h) should satisfy M * sigma * G3 = -G3
# (M is the ISD projector with eigenvalue -1 for ISD fluxes)

# For integer flux we check how well this is satisfied
G3 = f_int - tau_ex * h_ex_f
sigma_G3 = sigma @ G3
M_sigma_G3 = M0_ex @ sigma_G3

print("Self-duality check: M σ G₃ ≈ -G₃ ?")
print(f"  G₃         = {np.array(G3.real).round(3)} + i * {np.array(G3.imag).round(3)}")
print(f"  M σ G₃     = {np.array(M_sigma_G3.real).round(3)} + i * {np.array(M_sigma_G3.imag).round(3)}")
print(f"  -G₃        = {np.array(-G3.real).round(3)} + i * {np.array(-G3.imag).round(3)}")
print()
# For the continuous flux this should be exact
G3_cont = f_ex_cont - tau_ex * h_ex_f
sigma_G3_cont = sigma @ G3_cont
M_sigma_G3_cont = M0_ex @ sigma_G3_cont
print(f"  ||M σ G₃_cont - (-G₃_cont)|| = {float(jnp.linalg.norm(M_sigma_G3_cont + G3_cont)):.2e}  (continuous, should be ~0)")

<a id="7"></a>
## 6. Step 6 — Refinement with `scipy.optimize.root`

The ISD-completed, integer-rounded fluxes are **not** exact SUSY vacua: after rounding, $D_I W \neq 0$. The final step is to numerically solve

$$
    D_I W(z, \tau) = 0
$$

for the moduli and axio-dilaton, holding the integer flux fixed.

Rather than using the Newton method built into `jaxvacua`, we use **`scipy.optimize.root`** (Hybr method by default) for transparency. We pack the real and imaginary parts of $(z_1, z_2, \tau)$ into a real vector, evaluate $D_I W$ via the JAX model, and let SciPy find the root.

### Initial guess strategy

The moduli point at which ISD completion was performed is an excellent initial guess: the rounded integer flux is close to ISD at that point, so $D_I W$ is small there.

In [ ]:
# Run the full sample_bounded_fluxes with refine=False to get initial guesses
bf2 = bounded_fluxes(model, sampler=sampler, Nmax=NMAX)

candidates = bf2.sample_bounded_fluxes(
    n_target=5_000,
    n_batch=100_000,
    n_sample=1_000,
    n_mod=1_000,
    max_batches=20,
    verbose=True,
    seed=42,
    refine=False,
    return_moduli=True,
)
print(f"\nFound {len(candidates)} initial candidates.")

In [ ]:
# Inspect the F-term residuals at the initial guess moduli points
print("F-term residuals |DW| at initial guess moduli:")
print(f"{'Flux':>12s}  {'|DW| pre-refinement':>22s}  N_flux")
print("-" * 60)
for r in candidates:
    z   = jnp.array(r["moduli"], dtype=complex)
    tau = jnp.array(r["tau"], dtype=complex)
    fl  = jnp.array(r["flux"])
    DW  = model.DW(z, jnp.conj(z), tau, jnp.conj(tau), fl)
    nfl = float(model.tadpole(fl).real)
    residual = float(jnp.linalg.norm(DW))
    if residual < 2:  # only show those with reasonably small residuals
        print(f"  {str(r['flux'].astype(int).tolist()):>12s}  {residual:>22.6e}  {nfl:.0f}")

In [ ]:
# Define the root-finding problem for scipy.optimize.root
# Pack (Re(z1), Im(z1), Re(z2), Im(z2), Re(tau), Im(tau)) into a real 6-vector x

h12 = model.h12  # number of complex structure moduli

def pack(z, tau):
    """Complex moduli + tau -> real vector."""
    z_np = np.array(z)
    return np.concatenate([z_np.real, z_np.imag, [tau.real, tau.imag]])

def unpack(x):
    """Real vector -> (z_jax, tau_jax)."""
    z_re  = jnp.array(x[:h12])
    z_im  = jnp.array(x[h12:2*h12])
    z     = z_re + 1j * z_im
    tau   = x[2*h12] + 1j * x[2*h12 + 1]
    return z, tau

def residual_fn(x, flux):
    """F-term residual Re(DW) concatenated with Im(DW), as a real vector."""
    z, tau = unpack(x)
    DW = model.DW(z, jnp.conj(z), jnp.array(tau), jnp.conj(jnp.array(tau)), flux)
    return np.concatenate([np.array(DW.real), np.array(DW.imag)])

# Quick sanity check on a known initial guess
r0   = candidates[0]
fl0  = jnp.array(r0["flux"])
x0   = pack(r0["moduli"], complex(r0["tau"]))
res0 = residual_fn(x0, fl0)
print(f"Residual vector at initial guess: {res0.round(4)}")
print(f"||residual|| = {np.linalg.norm(res0):.4e}")

In [ ]:
# Solve DW = 0 for each candidate using scipy.optimize.root
# DW has h12+1 = 3 complex components → 6 real equations
# Variables: (Re(z1), Im(z1), Re(z2), Im(z2), Re(tau), Im(tau)) → 6 real unknowns

TOL_SCIPY = 1e-10

scipy_results = []
for r in tqdm(candidates):
    fl   = jnp.array(r["flux"])
    x0   = pack(r["moduli"], complex(r["tau"]))

    sol  = root(residual_fn, x0, args=(fl,), method="hybr", tol=TOL_SCIPY)
    z_sol, tau_sol = unpack(sol.x)

    DW_final = residual_fn(sol.x, fl)
    res_final = np.linalg.norm(DW_final)
    converged = sol.success and res_final < TOL_SCIPY * 1e3  # slightly loose

    scipy_results.append({
        "flux":       r["flux"],
        "moduli":     np.array(z_sol),
        "tau":        complex(tau_sol),
        "residual":   res_final,
        "converged":  converged,
        "scipy_msg":  sol.message,
    })

n_conv = sum(r["converged"] for r in scipy_results)
print(f"scipy.optimize.root: {n_conv}/{len(scipy_results)} converged (tol={TOL_SCIPY:.0e})")

In [ ]:
# Filter converged solutions and check whether they lie inside the sampler's moduli patch.
# Patch conditions: Im(z_i) ∈ [moduli_lower, moduli_upper], s ∈ [s_lower, s_upper], c₀ ∈ [axion_lower, axion_upper]

def in_patch(z, tau):
    im_z = np.imag(z)
    s    = np.imag(tau)
    c0   = np.real(tau)
    return (
        np.all(im_z >= sampler.moduli_lower) and
        np.all(im_z <= sampler.moduli_upper) and
        s >= sampler.s_lower and s <= sampler.s_upper and
        c0 >= sampler.axion_lower and c0 <= sampler.axion_upper
    )

print(f"{'Flux':>45s}  {'|DW|':>12s}  {'in patch':>10s}  {'N_flux':>8s}")
print("-" * 90)
n_in_patch = 0
vacua = []
seen  = set()
for r in scipy_results:
    if not r["converged"]:
        continue
    key = tuple(int(x) for x in r["flux"])
    if key in seen:
        continue
    seen.add(key)
    patch = in_patch(r["moduli"], r["tau"])
    if patch:
        n_in_patch += 1
        vacua.append(r)
    fl_str = str(r["flux"].astype(int).tolist())
    nfl    = float(model.tadpole(jnp.array(r["flux"])).real)
    print(f"{fl_str:>45s}  {r['residual']:>12.2e}  {'yes' if patch else 'no':>10s}  {nfl:>8.0f}")

print(f"\nConverged and in-patch: {n_in_patch} unique vacua")

In [ ]:
# Print the refined vacua in detail
if vacua:
    print(f"{'='*80}")
    print(f"Found {len(vacua)} refined SUSY vacua:")
    print(f"{'='*80}")
    for i, r in enumerate(vacua):
        z   = jnp.array(r["moduli"], dtype=complex)
        tau = jnp.array(r["tau"], dtype=complex)
        fl  = jnp.array(r["flux"])
        DW  = model.DW(z, jnp.conj(z), tau, jnp.conj(tau), fl)
        nfl = float(model.tadpole(fl).real)

        print(f"\nVacuum {i+1}:")
        print(f"  flux    = {r['flux'].astype(int).tolist()}")
        print(f"  moduli  = {np.array(r['moduli'])}")
        print(f"  tau     = {r['tau']:.8f}")
        print(f"  N_flux  = {nfl:.0f}")
        print(f"  |DW|    = {r['residual']:.2e}")
        print(f"  DW_vec  = {np.array(DW).round(12)}")
else:
    print("No converged in-patch vacua found in this run. Try increasing n_target or n_batch.")

### Comparison: scipy vs Newton

Let us verify that `scipy.optimize.root` and the built-in Newton method converge to the same vacua.

In [ ]:
# Re-run sample_bounded_fluxes with Newton refinement (default step_size=1.0)
bf3 = bounded_fluxes(model, sampler=sampler, Nmax=NMAX)

newton_vacua = bf3.sample_bounded_fluxes(
    n_target=50,
    n_batch=100_000,
    n_sample=300,
    n_mod=15,
    max_batches=10,
    verbose=True,
    seed=42,
    refine=True,
    newton_step_size=1.0,  # full Newton steps, quadratic convergence
    newton_tol=1e-10,
    newton_max_iters=30,
)
print(f"Newton refinement found {len(newton_vacua)} vacua.")
print(f"scipy.optimize.root found {len(vacua)} vacua.")

In [ ]:
# Compare flux vectors found by both methods
scipy_flux_set  = {tuple(int(x) for x in r["flux"]) for r in vacua}
newton_flux_set = {tuple(int(x) for x in r["flux"]) for r in newton_vacua}

print(f"Flux vectors in scipy result:  {len(scipy_flux_set)}")
print(f"Flux vectors in Newton result: {len(newton_flux_set)}")
print(f"Intersection: {len(scipy_flux_set & newton_flux_set)}")
print(f"Only in scipy:  {scipy_flux_set - newton_flux_set}")
print(f"Only in Newton: {newton_flux_set - scipy_flux_set}")

In [ ]:
# For fluxes found by both, compare the converged moduli
if scipy_flux_set & newton_flux_set:
    common_key = next(iter(scipy_flux_set & newton_flux_set))

    r_scipy  = next(r for r in vacua       if tuple(int(x) for x in r["flux"]) == common_key)
    r_newton = next(r for r in newton_vacua if tuple(int(x) for x in r["flux"]) == common_key)

    print(f"Flux: {list(common_key)}")
    print(f"  scipy  moduli = {r_scipy['moduli']}")
    print(f"  Newton moduli = {r_newton['moduli']}")
    print(f"  scipy  tau    = {r_scipy['tau']:.10f}")
    print(f"  Newton tau    = {r_newton['tau']:.10f}")
    print(f"  ||Δmoduli||   = {np.linalg.norm(r_scipy['moduli'] - r_newton['moduli']):.2e}")
    print(f"  |Δtau|        = {abs(r_scipy['tau'] - r_newton['tau']):.2e}")

<a id="8"></a>
## 7. Summary and consistency checks

We now perform a final self-consistency check on each refined vacuum.

In [ ]:
def check_vacuum(r, model, sampler, label=""):
    """Print a consistency table for one refined vacuum."""
    z   = jnp.array(r["moduli"], dtype=complex)
    tau = jnp.array(r["tau"], dtype=complex)
    fl  = jnp.array(r["flux"])

    DW   = model.DW(z, jnp.conj(z), tau, jnp.conj(tau), fl)
    nfl  = float(model.tadpole(fl).real)
    M0_v = model.ISD_matrix(z, jnp.conj(z))
    G3   = fl[:model.n_fluxes] - tau * fl[model.n_fluxes:]
    isd_res = float(jnp.linalg.norm(M0_v @ (model.periods.sigma @ G3) + G3))

    patch = in_patch(np.array(z), complex(tau))

    header = f"--- Vacuum {label} ---"
    print(header)
    print(f"  flux            = {np.array(fl).astype(int).tolist()}")
    print(f"  moduli Im(z)    = {np.imag(np.array(z))}")
    print(f"  moduli Re(z)    = {np.real(np.array(z))}")
    print(f"  tau             = {complex(tau):.8f}")
    print(f"  N_flux          = {nfl:.2f}  (should be integer in (0,{NMAX}])")
    print(f"  ||DW||          = {float(jnp.linalg.norm(DW)):.2e}  (should be < 1e-8)")
    print(f"  ||M σ G3 + G3|| = {isd_res:.2e}  (ISD residual; 0 = exact ISD)")
    print(f"  in sampler patch= {patch}")
    print()

if vacua:
    for i, r in enumerate(vacua[:3]):
        check_vacuum(r, model, sampler, label=str(i+1))
else:
    print("No vacua to check. Increase n_target or n_batch.")

In [ ]:
# Tadpole and moduli distribution of all refined vacua
all_refined = vacua + newton_vacua  # combine both refinement methods
# Deduplicate by flux
seen_all = set()
all_refined_dedup = []
for r in all_refined:
    key = tuple(int(x) for x in r["flux"])
    if key not in seen_all:
        seen_all.add(key)
        all_refined_dedup.append(r)

print(f"Total unique refined vacua: {len(all_refined_dedup)}")

if len(all_refined_dedup) >= 2:
    tads = np.array([float(model.tadpole(jnp.array(r["flux"])).real) for r in all_refined_dedup])
    mods = np.array([r["moduli"] for r in all_refined_dedup])
    taus = np.array([r["tau"] for r in all_refined_dedup])

    fig, axes = plt.subplots(1, 3, dpi=130, figsize=(13, 3.5))

    axes[0].hist(tads, bins=range(0, NMAX + 2), edgecolor='k', lw=0.5)
    axes[0].set_xlabel(r'$N_{\rm flux}$', fontsize=12)
    axes[0].set_ylabel('Count', fontsize=12)
    axes[0].set_title('Tadpole distribution', fontsize=11)

    sc = axes[1].scatter(
        mods[:, 0].imag, mods[:, 1].imag,
        c=tads, cmap='viridis', s=20, alpha=0.8
    )
    plt.colorbar(sc, ax=axes[1], label=r'$N_{\rm flux}$')
    axes[1].set_xlabel(r'$\mathrm{Im}(z_1)$', fontsize=12)
    axes[1].set_ylabel(r'$\mathrm{Im}(z_2)$', fontsize=12)
    axes[1].set_title('Moduli distribution', fontsize=11)

    axes[2].scatter(taus.real, taus.imag, c=tads, cmap='viridis', s=20, alpha=0.8)
    axes[2].set_xlabel(r'$\mathrm{Re}(\tau)$', fontsize=12)
    axes[2].set_ylabel(r'$\mathrm{Im}(\tau) = s$', fontsize=12)
    axes[2].set_title(r'Axio-dilaton $\tau$', fontsize=11)

    plt.tight_layout()
    plt.show()
else:
    print("Not enough vacua to plot — try larger n_target.")

## Summary

This notebook deconstructed `sample_bounded_fluxes` into its individual steps:

| Step | What it does | Key formula |
|------|--------------|-------------|
| 1 | JIT-vmap eigenvalues over moduli sample | $\lambda_{\max}(\mathcal{M})$, $\mu_{\min}(-\mathrm{Im}\mathcal{N})$, $\tilde\mu_{\min}(\mathrm{Im}\mathcal{N}^{-1})$ |
| 2 | Bounding box with correct $s_{\min}$ | $\|h_1\|^2 \leq N_{\max}/(s_{\min}\,\tilde\mu_{\min})$, etc. |
| 3 | ISD completion $h \mapsto f$ | $f = s\,\mathcal{M}\,\Sigma\,h + c_0\,h$ (mode `"H"`, consistent with `sampling.py`) |
| 4 | Tadpole + eigenvalue filters | $N_{\mathrm{flux}} = |f^T\Sigma h| \in (0, N_{\max}]$ |
| 5 | ISD convention cross-check | `flux_bounding.py` = `sampling.py._ISD_sampling_FH` |
| 6 | Refinement via `scipy.optimize.root` | Solve $D_I W(z,\tau) = 0$ for integer flux |

### Key lesson: `dil_min` must match the sampler

Using the SL(2,Z) fundamental domain floor $s_{\min} = \sqrt{3}/2 \approx 0.866$ when the sampler enforces $s \geq 2$ inflates the bounding box volume by $(2/0.866)^4 \approx 40\times$, making most randomly sampled $h$ vectors violate the tadpole constraint. The fix — automatically reading `sampler.s_lower` — is critical for achieving non-trivial candidate yield.